# Models and training

The second of three notebooks. The first covers the data pipeline and the metric, this
one the models and how they are trained, the third the evaluation and the results.

The properties that the argument of the project rests on are demonstrated here rather
than asserted: locality of the cellular rule, the global receptive field of the
comparison model, and the absence of spurious gradients at the room border. All of them
are checked against quantities that can be verified independently of the code that
produces them.

All logic lives in `src/`; this notebook only calls it. A short training run is included
to show the mechanics, but it is not the model used for the results.

In [ ]:
import sys

sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

from src.damage.stochastic import erasure, tile_flip
from src.models.aux_targets import access_distance_field
from src.models.encoding import decode, to_nca_state, visible_channels
from src.models.nca import NCA, _perception_filters
from src.models.unet import UNet
from src.tiles import CHAR_MAP, NUM_TILES
from src.train.pool import SamplePool
from src.train.trainer import Trainer
from src.viz import parse_room, render_room

plt.rcParams.update({"figure.dpi": 130, "font.size": 9})
torch.manual_seed(0)

rooms = np.load("../data/processed/rooms.npz", allow_pickle=True)["rooms"]
room = rooms[0]
print(f"{len(rooms)} unique rooms of shape {room.shape}")

## 1. State encoding

A cell does not hold a tile, it holds a vector. Ten visible channels carry a one-hot
encoding of the tile; twelve hidden channels start at zero and carry whatever the
training decides to put there. The hidden channels are the only way information can
travel across the grid, and nothing in the loss constrains what they contain.

In [ ]:
state = to_nca_state(torch.as_tensor(rooms[:1]), hidden_channels=12)
print(f"room  {rooms[0].shape}  ->  state {tuple(state.shape)}")
print(f"      {NUM_TILES} visible + {state.shape[1] - NUM_TILES} hidden channels per cell")

# the visible channels are one-hot, so they decode back to the original tiles
print(f"decode(state) == room: {torch.equal(decode(state)[0], torch.as_tensor(rooms[0]))}")
print(f"hidden channels are zero: {bool((state[:, NUM_TILES:] == 0).all())}")

## 2. Perception uses fixed filters

The cell does not learn what to look at. Three fixed filters are applied to every
channel independently: the identity, and the two Sobel operators, which are the
discrete spatial derivatives along each axis. Each cell therefore receives its own
value plus the rate of change towards its neighbours, which is what gives it a sense
of direction.

The filters are registered as a buffer, not as parameters: they never receive a
gradient. This is a choice inherited from the original formulation, where the rule is
meant to hold the learned behaviour and the perception is given.

In [ ]:
filters = _perception_filters(1)
names = ["identity", "sobel x", "sobel y"]

fig, axes = plt.subplots(1, 3, figsize=(4.5, 1.6))
for ax, f, name in zip(axes, filters, names):
    ax.imshow(f[0], cmap="RdBu_r", vmin=-0.3, vmax=0.3)
    for (i, j), v in np.ndenumerate(f[0].numpy()):
        ax.text(j, i, f"{v:.3g}", ha="center", va="center", fontsize=6)
    ax.set_title(name, fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()

nca = NCA(hidden_channels=12)
print(f"filter bank: {tuple(nca.filters.shape)}")
print(f"  {nca.filters.shape[0] // nca.num_channels} filters x {nca.num_channels} channels")
print(f"  second dimension is 1: each filter reads a single channel (groups={nca.num_channels})")
print(f"\ntrainable parameters: {sum(p.numel() for p in nca.parameters()):,}")
print(f"filters among them: {any(p is nca.filters for p in nca.parameters())}")

## 3. Border padding, verified against an external property

A Sobel filter sums to zero, so its response on a constant region must be exactly zero.
That is a property of the operator, independent of the implementation, which makes it a
usable check.

With zero padding the property fails at the border: the convolution sees a transition
from the content to the padding and reports an edge that does not exist. The original
formulation grows a figure inside a larger canvas, where the border is empty and this
never matters. Rooms fill the grid to the perimeter, and the perimeter cells are exactly
the ones that define the enclosure, so the choice matters here.

In [ ]:
import torch.nn.functional as F

constant = torch.ones(1, nca.num_channels, 8, 8)

zero_pad = F.conv2d(F.pad(constant, (1, 1, 1, 1), mode="constant", value=0.0),
                    nca.filters, groups=nca.num_channels)
repl_pad = nca.perceive(constant)

def sobel_energy(perceived):
    per_channel = perceived.reshape(1, nca.num_channels, 3, 8, 8)
    return float(per_channel[:, :, 1:3].abs().sum())

print(f"sobel response on a constant region")
print(f"  zero padding:        {sobel_energy(zero_pad):.4f}")
print(f"  replicate padding:   {sobel_energy(repl_pad):.4f}")

## 4. Locality

The property the whole project turns on. Perturbing one cell and taking a single step
must leave every cell outside its immediate neighbourhood untouched. This is checked by
measuring rather than asserted from the architecture.

In [ ]:
probe = NCA(hidden_channels=12)
probe.update_prob = 1.0                      # remove sampling noise from the measurement
nn.init.normal_(probe.w2.weight, std=0.1)    # the head starts at zero, so give it a delta

s1 = to_nca_state(torch.as_tensor(rooms[:1]), 12)
s2 = s1.clone()
s2[0, :, 5, 7] += 5.0

difference = (probe(s1) - probe(s2)).abs().sum(1)[0]
changed = (difference > 1e-6)

outside = changed.clone()
outside[4:7, 6:9] = False
print(f"cells changed in total:              {int(changed.sum())}")
print(f"cells changed outside the 3x3 patch: {int(outside.sum())}")

fig, ax = plt.subplots(figsize=(3.2, 2.2))
ax.imshow(difference.detach(), cmap="inferno")
ax.add_patch(plt.Rectangle((5.5, 3.5), 3, 3, fill=False, edgecolor="#4ad", lw=1.4))
ax.set_title("effect of perturbing cell (5, 7)", fontsize=8)
ax.set_xticks([]); ax.set_yticks([])

## 5. The update is residual and starts from the identity

The rule produces an increment rather than a new state, which gives the gradient an
identity path through time and is the main defence against the depth of the unrolled
recurrence. The head is initialised at zero, so before any training the increment is
zero and the first step leaves the state untouched: the model starts from doing nothing
and learns how far to depart from it.

This does not stall learning. The gradient with respect to the head depends on the
activations feeding it, not on the head itself, so it is non-zero from the first
iteration.

In [ ]:
fresh = NCA(hidden_channels=12)
s = to_nca_state(torch.as_tensor(rooms[:1]), 12)
print(f"untrained model, first step is the identity: {torch.allclose(fresh(s), s)}")

# with a non-zero head, output minus input equals the increment computed by hand
probe.update_prob = 1.0
delta = probe.w2(F.relu(probe.w1(probe.perceive(s1))))
print(f"output - input equals the computed increment: "
      f"{torch.allclose(probe(s1) - s1, delta, atol=1e-6)}")

# gradient reaches the parameters through the unrolled steps
x = to_nca_state(torch.as_tensor(rooms[:1]), 12).requires_grad_(True)
for _ in range(8):
    x = fresh(x)
x.sum().backward()
reached = [n for n, p in fresh.named_parameters() if p.grad is not None and p.grad.abs().sum() > 0]
print(f"parameters receiving gradient after 8 steps: {reached}")

## 6. Damage applied during training

Two stochastic corruptions. Erasure zeroes a contiguous patch, leaving cells with no
tile at all; tile flip replaces scattered cells with a wrong tile. The second is harder:
the model has to recognise that the structure is wrong before correcting it, rather than
filling an obvious hole.

Both zero every channel of the affected cells, hidden ones included, so that a damaged
cell never carries a visible tile that contradicts its hidden state.

Targeted damage is never used during training. It lives in a separate module and appears
only at evaluation time, which is what makes a repaired door evidence of generalisation
rather than of memorisation.

In [ ]:
rng = np.random.default_rng(0)
clean = to_nca_state(torch.as_tensor(rooms[:1]), 12)

eras, mask_e = erasure(clean, rng, fraction=0.25)
flip, mask_f = tile_flip(clean, np.random.default_rng(0), fraction=0.15)

fig, axes = plt.subplots(1, 3, figsize=(7, 1.9))
render_room(rooms[0], axes[0], "pristine")
render_room(decode(eras)[0].numpy(), axes[1], "erasure, 25 percent",
            highlight=mask_e[0, 0].numpy())
render_room(decode(flip)[0].numpy(), axes[2], "tile flip, 15 percent",
            highlight=mask_f[0, 0].numpy())
plt.tight_layout()

print(f"erased cells keep no channel active: {bool((eras[:, :, mask_e[0, 0]] == 0).all())}")

## 7. The sample pool

The pool does not hold rooms, it holds states in mid-evolution. Every slot stores a
state that keeps changing and the target room it should converge to, and the target is
what allows a single unconditioned model to handle many different rooms.

All slots start identical, at step zero. The diversity is not designed: it accumulates,
because slots are drawn at random, damaged or not, evolved for a variable number of
steps, and written back. After a few hundred iterations the pool holds freshly damaged
states, half-repaired ones and states that have been stable for a while.

Without it, the model would only ever see the trajectory that starts from a clean room.
The pool forces it to recover from states it produced itself, including poor ones, and
that is what turns the target into a stable attractor rather than a point reached once.

In [ ]:
pool = SamplePool(rooms, pool_size=1024, hidden_channels=12, device="cpu", seed=0)
print(f"slots: {pool.pool_size}, distinct rooms: {len(rooms)}")
print(f"state  {tuple(pool.state.shape)}")
print(f"target {tuple(pool.target.shape)}")
print(f"slots per room on average: {pool.pool_size / len(rooms):.1f}")

slots, states, targets = pool.sample(8)
print(f"\na batch: slots {tuple(slots.shape)}, states {tuple(states.shape)}, "
      f"targets {tuple(targets.shape)}")

## 8. A training iteration

Draw a batch, replace the worst state with a clean room so that the pool does not drift
into degenerate configurations, damage part of the batch, unroll for a variable number
of steps, compare against the target and write the evolved states back.

The loss is a cross-entropy over the ten visible channels. A tile is a category, not a
quantity, so a squared error would make an arbitrary index distance meaningful. The
hidden channels have no target at all, except in the multitask variant of section 11.

Note that the loss is computed once, at the end of the unroll, and the gradient travels
back through every step.

In [ ]:
demo_pool = SamplePool(rooms, pool_size=256, hidden_channels=12, device="cpu", seed=0)
demo = Trainer(NCA(hidden_channels=12), demo_pool,
               lr=1e-3, grad_clip=1.0, bptt_min=16, bptt_max=24, batch_size=8,
               damage_prob=0.5, damage_fractions=[0.2, 0.4], device="cpu", seed=0)

print(f"unroll drawn from [{demo.bptt_min}, {demo.bptt_max}] at every iteration")
print(f"damage applied to {demo.damage_prob:.0%} of the batch")
print(f"loss over the {NUM_TILES} visible channels only")

## 9. A short training run

Four hundred iterations on CPU, with a shorter unroll than the real setting. This is
enough to show that the loss falls and the mechanics work, and far from enough to
produce a usable model: the reported runs are eight thousand iterations with an unroll
between 64 and 96, and the checkpoints they produce are what the third notebook reads.

The starting value is well below the value of a uniform guess over ten tiles, because
the initial state already encodes the target room. This is worth keeping in mind: the
loss is averaged over all cells, most of which are undamaged, so it can look healthy
while repair fails. That is the same observation that motivates the topological metric.

In [ ]:
import time

history = []
t0 = time.time()
for i in range(400):
    history.append(demo.train_step())

print(f"{len(history)} iterations in {time.time() - t0:.0f} s")
print(f"uniform guess over {NUM_TILES} tiles: {np.log(NUM_TILES):.3f}")
print(f"first 50 iterations:  {np.mean(history[:50]):.3f}")
print(f"last 50 iterations:   {np.mean(history[-50:]):.3f}")

smooth = np.convolve(history, np.ones(20) / 20, mode="valid")
fig, ax = plt.subplots(figsize=(5.4, 2.6))
ax.plot(history, lw=0.4, alpha=0.35)
ax.plot(range(19, len(history)), smooth, lw=1.6)
ax.set_xlabel("iteration")
ax.set_ylabel("cross-entropy")
ax.grid(alpha=0.3)

In [ ]:
# what the demo model does with a damaged room after 400 iterations
demo.nca.eval()
target = torch.as_tensor(rooms[:1])
with torch.no_grad():
    damaged, dmg_mask = erasure(to_nca_state(target, 12), np.random.default_rng(3), 0.2)
    evolved = damaged.clone()
    snapshots = {0: decode(evolved)[0].numpy()}
    for step in range(1, 49):
        evolved = demo.nca(evolved)
        if step in (8, 24, 48):
            snapshots[step] = decode(evolved)[0].numpy()

fig, axes = plt.subplots(1, 5, figsize=(9, 1.9))
render_room(rooms[0], axes[0], "target")
for ax, (step, img) in zip(axes[1:], sorted(snapshots.items())):
    render_room(img, ax, f"step {step}")
plt.tight_layout()

## 10. The comparison model has a global receptive field

The U-Net exists to isolate one variable. It shares the interface, the pool, the loss
and the metric with the cellular model; only the receptive field differs.

The same measurement used for locality, applied to it, gives the opposite answer:
perturbing a corner cell changes the opposite corner. Two pooling stages put every
bottleneck cell in contact with a region larger than the room, so a single forward pass
covers what the cellular rule needs at least sixteen steps to traverse.

In [ ]:
unet = UNet(hidden_channels=12)
nn.init.normal_(unet.head.weight, std=0.1)

u1 = to_nca_state(torch.as_tensor(rooms[:1]), 12)
u2 = u1.clone()
u2[0, :, 0, 0] += 5.0
u_diff = (unet(u1) - unet(u2)).abs().sum(1)[0].detach()

print(f"NCA   parameters: {sum(p.numel() for p in nca.parameters()):>9,}")
print(f"U-Net parameters: {sum(p.numel() for p in unet.parameters()):>9,}")
print(f"\neffect of perturbing cell (0, 0) on the opposite corner:")
nca_corner = (probe(s1) - probe(s2)).abs().sum(1)[0][-1, -1].detach()
print(f"  NCA   (one step): {float(nca_corner):.6f}")
print(f"  U-Net (one pass): {float(u_diff[-1, -1]):.6f}")

fig, ax = plt.subplots(figsize=(3.2, 2.2))
ax.imshow(u_diff, cmap="inferno")
ax.set_title("U-Net: effect of perturbing cell (0, 0)", fontsize=8)
ax.set_xticks([]); ax.set_yticks([])

## 11. The auxiliary topological signal

For the multitask variant, one hidden channel is supervised towards the geodesic
distance to the nearest access point, computed on the pristine room by breadth-first
search over walkable cells and normalised by a fixed constant so that the value means
the same thing in every room.

The signal is chosen because it is global, in that it depends on the whole room, while
still being expressible as a per-cell field. That is precisely the kind of quantity a
local rule ought to be able to propagate, if it can propagate anything at all.

No channel is added: an existing hidden channel is supervised, so the multitask model
has exactly the same architecture and the same parameter count as the baseline and the
loss term is the only difference. The field comes from the pristine room while the input
is damaged, so the model has to infer the correct topology rather than copy it. At
evaluation time the hidden channels are not decoded, so the metric is unaffected.

In [ ]:
field = access_distance_field(rooms[0])

fig, axes = plt.subplots(1, 2, figsize=(5.5, 2))
render_room(rooms[0], axes[0], "room")
im = axes[1].imshow(field, cmap="viridis")
axes[1].set_title("distance to the nearest access", fontsize=8)
axes[1].set_xticks([]); axes[1].set_yticks([])
fig.colorbar(im, ax=axes[1], fraction=0.03)
plt.tight_layout()

acc = (rooms[0] == CHAR_MAP["D"]) | (rooms[0] == CHAR_MAP["S"])
print(f"value at the access points: {field[acc].max():.3f}")
print(f"value on unreachable cells: {field[~acc].max():.3f}")

The training entry point selects the architecture through the configuration, so all
three models go through the same pool, trainer and evaluation code:

```
python scripts/train.py                                   # NCA
python scripts/train.py model=unet model.iterative=false \
       train.bptt_min=1 train.bptt_max=1                   # U-Net, one-shot
python scripts/train.py train.aux_weight=1.0               # NCA, multitask
```

The third notebook reads the checkpoints these produce.